In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install ultralytics gradio requests -q

import requests

# Download weights from GitHub
url = "https://github.com/amoghsjadhav/PlateCalc/raw/main/best.pt"
response = requests.get(url)
with open('/kaggle/working/best.pt', 'wb') as f:
    f.write(response.content)
print("Weights downloaded:", len(response.content) / 1e6, "MB")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.7 MB/s eta 0:00:00
Weights downloaded: 0.311461 MB


In [2]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.pt'):
            print(os.path.join(root, file))

/kaggle/input/datasets/amgdotexe/platecalc-weights/best.pt


In [3]:
from ultralytics import YOLO

model = YOLO('/kaggle/input/datasets/amgdotexe/platecalc-weights/best.pt')
print("Model loaded successfully")
print("Sample classes:", [model.names[i] for i in range(5)])

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model loaded successfully
Sample classes: ['candy', 'egg tart', 'french fries', 'chocolate', 'biscuit']


In [4]:
import os

# Grab a test image from the dataset we already have
test_img = '/kaggle/input/datasets/ggrill/foodseg103/FoodSeg103/Images/img_dir/test/00007111.jpg'

results = model(test_img)

print("Detected items:")
for box in results[0].boxes:
    class_id = int(box.cls)
    confidence = float(box.conf)
    print(f"  {model.names[class_id]}: {confidence:.0%} confidence")


image 1/1 /kaggle/input/datasets/ggrill/foodseg103/FoodSeg103/Images/img_dir/test/00007111.jpg: 480x640 (no detections), 334.3ms
Speed: 11.7ms preprocess, 334.3ms inference, 8.0ms postprocess per image at shape (1, 3, 480, 640)
Detected items:


In [5]:
import os

# Try a few different images
test_images = os.listdir('/kaggle/input/datasets/ggrill/foodseg103/FoodSeg103/Images/img_dir/test')[:5]

for img_file in test_images:
    img_path = f'/kaggle/input/datasets/ggrill/foodseg103/FoodSeg103/Images/img_dir/test/{img_file}'
    results = model(img_path, conf=0.1)  # lower confidence threshold
    detections = len(results[0].boxes)
    print(f"{img_file}: {detections} detections")


image 1/1 /kaggle/input/datasets/ggrill/foodseg103/FoodSeg103/Images/img_dir/test/00007111.jpg: 480x640 1 cheese butter, 1 shrimp, 1 potato, 1 carrot, 152.2ms
Speed: 2.3ms preprocess, 152.2ms inference, 23.2ms postprocess per image at shape (1, 3, 480, 640)
00007111.jpg: 4 detections

image 1/1 /kaggle/input/datasets/ggrill/foodseg103/FoodSeg103/Images/img_dir/test/00005029.jpg: 448x640 1 cake, 1 strawberry, 3 cherrys, 160.4ms
Speed: 2.2ms preprocess, 160.4ms inference, 6.9ms postprocess per image at shape (1, 3, 448, 640)
00005029.jpg: 5 detections

image 1/1 /kaggle/input/datasets/ggrill/foodseg103/FoodSeg103/Images/img_dir/test/00006166.jpg: 640x640 1 ice cream, 4 cakes, 1 apple, 1 peach, 226.4ms
Speed: 2.6ms preprocess, 226.4ms inference, 13.3ms postprocess per image at shape (1, 3, 640, 640)
00006166.jpg: 7 detections

image 1/1 /kaggle/input/datasets/ggrill/foodseg103/FoodSeg103/Images/img_dir/test/00004975.jpg: 480x640 1 chicken duck, 1 cauliflower, 1 carrot, 1 celery stick, 1 

In [6]:
COUNTABLE_FOODS = {
    'egg', 'apple', 'banana', 'orange', 'pear', 'peach',
    'kiwi', 'mango', 'cherry', 'strawberry', 'grape',
    'dumpling', 'cookie', 'meatball'
}

UNIT_WEIGHTS = {
    'egg': 50, 'apple': 182, 'banana': 118, 'orange': 131,
    'pear': 166, 'peach': 150, 'kiwi': 69, 'mango': 200,
    'cherry': 8, 'strawberry': 12, 'grape': 5,
    'dumpling': 20, 'cookie': 14, 'meatball': 30
}

print("Countable foods defined:", len(COUNTABLE_FOODS))

Countable foods defined: 14


In [7]:
import requests

def get_calories_per_100g(food_name: str) -> float:
    url = "https://api.nal.usda.gov/fdc/v1/foods/search"
    params = {
        "query": food_name,
        "pageSize": 1,
        "api_key": "DEMO_KEY"
    }
    try:
        resp = requests.get(url, params=params, timeout=5).json()
        foods = resp.get("foods", [])
        if not foods:
            return 0
        for nutrient in foods[0].get("foodNutrients", []):
            if nutrient.get("nutrientName") == "Energy":
                return nutrient.get("value", 0)
    except:
        return 0
    return 0

# Test it
print("Rice:", get_calories_per_100g("rice"), "kcal/100g")
print("Egg:", get_calories_per_100g("egg"), "kcal/100g")
print("Bread:", get_calories_per_100g("bread"), "kcal/100g")

Rice: 139 kcal/100g
Egg: 513 kcal/100g
Bread: 146 kcal/100g


In [8]:
import cv2
import numpy as np

def estimate_calories(image_path: str, model) -> dict:
    results = model(image_path)[0]
    total_calories = 0
    breakdown = []

    if results.masks is None:
        return {"total": 0, "items": [], "message": "No food detected"}

    img = cv2.imread(image_path)
    total_pixels = img.shape[0] * img.shape[1]

    for i, (mask, box) in enumerate(zip(results.masks.data, results.boxes)):
        class_id = int(box.cls)
        food_name = model.names[class_id]
        confidence = float(box.conf)

        if confidence < 0.3:  # skip low confidence detections
            continue

        mask_np = mask.cpu().numpy().astype(np.uint8)
        food_pixels = int(mask_np.sum())

        # Quantity estimation
        if food_name in COUNTABLE_FOODS:
            num_labels, _ = cv2.connectedComponents(mask_np)
            count = max(1, num_labels - 1)
            weight_g = count * UNIT_WEIGHTS.get(food_name, 100)
            method = f"{count} unit(s) × {UNIT_WEIGHTS.get(food_name, 100)}g"
        else:
            weight_g = (food_pixels / total_pixels) * 500
            method = f"{food_pixels} pixels → {weight_g:.0f}g"

        cal_per_100g = get_calories_per_100g(food_name)
        calories = (weight_g / 100) * cal_per_100g

        total_calories += calories
        breakdown.append({
            "food": food_name,
            "confidence": f"{confidence:.0%}",
            "weight_g": round(weight_g, 1),
            "cal_per_100g": cal_per_100g,
            "calories": round(calories, 1),
            "method": method
        })

    return {"total": round(total_calories, 1), "items": breakdown}

In [9]:
import gradio as gr

def analyze_plate(image):
    if image is None:
        return "Please upload an image"
    
    result = estimate_calories(image, model)
    
    if not result["items"]:
        return "No food detected in this image"
    
    output = f"### Total Calories: {result['total']} kcal\n\n"
    output += "| Food | Confidence | Weight | Cal/100g | Calories |\n"
    output += "|------|------------|--------|----------|----------|\n"
    
    for item in result["items"]:
        output += f"| {item['food']} | {item['confidence']} | {item['weight_g']}g | {item['cal_per_100g']} | {item['calories']} kcal |\n"
    
    return output

demo = gr.Interface(
    fn=analyze_plate,
    inputs=gr.Image(type="filepath", label="Upload food image"),
    outputs=gr.Markdown(label="Calorie Breakdown"),
    title="PlateCalc",
    description="Upload a photo of your meal to estimate calories"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://62bc826c49d79e1819.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



image 1/1 /tmp/gradio/6953fb2284ff79b4b74786da0234fb1a44978659e39d8c00478ac76234902685/Screenshot 2026-07-09 221128.png: 448x640 1 ice cream, 1 cheese butter, 1 pineapple, 1 steak, 2 breads, 162.9ms
Speed: 3.9ms preprocess, 162.9ms inference, 6.7ms postprocess per image at shape (1, 3, 448, 640)
Created dataset file at: .gradio/flagged/dataset1.csv
